In [161]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza3.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5] 
q = df[:, 6] 

#p[76:] -= 3
#q[76:] -= 3

err_p = df[:, 8]
err_q =  df[:, 7]



p = np.delete(p, [0, 62, 88] )
q = np.delete(q, [0, 62, 88]  )
err_p = np.delete(err_p, [0, 62, 88] )
err_q = np.delete(err_q, [0, 62, 88] )



#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0
Parametri stimati:
A = -1.0080348131154342 ± 0.2179267858590797
B = 1.9155972596739395 ± 0.15616923546674824
C = 200.7759102272441 ± 0.09546267995490164
Chi2 = 14473.759824928402
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      14473.8
NDf                       =          110
Edm                       =   2.5788e-07
NCalls                    =          138
A                         =     -1.00803   +/-   0.217927    
B                         =       1.9156   +/-   0.156169    
C                         =      200.776   +/-   0.0954627   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


## Fit separati a p = 400



In [162]:

p_cut = 400

model_fit_1 = ROOT.TF1('model_1', model, min(p), p_cut, 3)      # parte con p < 400
model_fit_1.SetParameters(par[0], par[1], par[2])
model_fit_1.SetParNames("A", "B", "C")
model_fit_1.SetLineColor(ROOT.kRed + 1)
model_fit_1.SetLineWidth(2)

model_fit_2 = ROOT.TF1('model_2', model, p_cut, max(p), 3)      # parte con p > 400
model_fit_2.SetParameters(par[0], par[1], par[2])
model_fit_2.SetParNames("A", "B", "C")
model_fit_2.SetLineColor(ROOT.kGreen + 2)
model_fit_2.SetLineWidth(2)

gr_split = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr_split.SetTitle("Fit separati; p; q")
gr_split.SetMarkerStyle(20)
gr_split.SetMarkerSize(0.7)
gr_split.SetMarkerColor(ROOT.kAzure + 2)
gr_split.SetLineColor(ROOT.kAzure + 3)
gr_split.SetLineWidth(1)

fit_res_1 = gr_split.Fit(model_fit_1, "SRN")
fit_res_2 = gr_split.Fit(model_fit_2, "SRN")

par_1 = model_fit_1.GetParameters()
par_2 = model_fit_2.GetParameters()
errors_1 = model_fit_1.GetParErrors()
errors_2 = model_fit_2.GetParErrors()
chi2_1, ndf_1 = fit_res_1.Chi2(), fit_res_1.Ndf()
chi2_2, ndf_2 = fit_res_2.Chi2(), fit_res_2.Ndf()


print(f"Fit 1 (p < {p_cut}):")
print(f"A = {par_1[0]} ± {errors_1[0]}")
print(f"B = {par_1[1]} ± {errors_1[1]}")
print(f"C = {par_1[2]} ± {errors_1[2]}")
print(f"Chi2 = {chi2_1}   ndf = {ndf_1}")
print()
print(f"Fit 2 (p > {p_cut}):")
print(f"A = {par_2[0]} ± {errors_2[0]}")
print(f"B = {par_2[1]} ± {errors_2[1]}")
print(f"C = {par_2[2]} ± {errors_2[2]}")
print(f"Chi2 = {chi2_2}   ndf = {ndf_2}")

print()
for i, nome in enumerate(["A", "B", "C"]):
    diff = np.abs(par_1[i] - par_2[i])
    sigma = np.sqrt(errors_1[i]**2 + errors_2[i]**2)
    if diff < 3*sigma:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> compatibili")
    else:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> NON compatibili")

chi2_tot = model_fit.GetChisquare()      
ndf_tot = model_fit.GetNDF()
chi2_split = chi2_1 + chi2_2            
ndf_split = ndf_1 + ndf_2
print()
print(f"Fit unico    : chi2 = {chi2_tot:.1f}  ndf = {ndf_tot}  chi2/ndf = {chi2_tot/ndf_tot:.1f}")
print(f"Due fit somma: chi2 = {chi2_split:.1f}  ndf = {ndf_split}  chi2/ndf = {chi2_split/ndf_split:.1f}")

Fit 1 (p < 400):
A = -15.262294849204446 ± 1.0531397284068873
B = -43.52858147317508 ± 1.614277550453714
C = 213.58846187850563 ± 0.6420717981888823
Chi2 = 1570.0126559201021   ndf = 73

Fit 2 (p > 400):
A = -8.673913251047976 ± 0.5508388831759221
B = -3.792254695035056 ± 0.28882201013707726
C = 204.4377094556519 ± 0.1970219687004865
Chi2 = 2611.9386060981014   ndf = 34

A: differenza = 6.588 (5.5 sigma) -> NON compatibili
B: differenza = 39.736 (24.2 sigma) -> NON compatibili
C: differenza = 9.151 (13.6 sigma) -> NON compatibili

Fit unico    : chi2 = 14473.8  ndf = 110  chi2/ndf = 131.6
Due fit somma: chi2 = 4182.0  ndf = 107  chi2/ndf = 39.1
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1570.01
NDf                       =           73
Edm                       =  1.23533e-07
NCalls                    =          111
A                         =     -15.2623   +/-   1.05314     
B                         =     -43.5286   +/-   1

In [ ]:
c_split = ROOT.TCanvas("c_split", "Fit separati", 800, 600)
c_split.SetGrid()

gr_split.Draw("AP")
model_fit_1.Draw("same")
model_fit_2.Draw("same")

leg_split = ROOT.TLegend(0.40, 0.72, 0.89, 0.89)
leg_split.AddEntry(gr_split, "Dati", "pe")
leg_split.AddEntry(model_fit_1, f"Fit p < {p_cut}: A = {par_1[0]:.1f} #pm {errors_1[0]:.1f}, #chi^{{2}}/ndf = {chi2_1:.0f}/{ndf_1}", "l")
leg_split.AddEntry(model_fit_2, f"Fit p > {p_cut}: A = {par_2[0]:.1f} #pm {errors_2[0]:.1f}, #chi^{{2}}/ndf = {chi2_2:.0f}/{ndf_2}", "l")
leg_split.Draw()

c_split.Draw()
c_split.SaveAs("fit_root_split.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_split


In [ ]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A NON è compatibile con 0
---
B è compatibile con 0


In [ ]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff



#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Fit status: 0
Parametri stimati:
const = 44.718975457871366 ± 6.628934607212528
#mu = 1.671457946528047 ± 0.45590938250607677
#sigma = 4.1082096969150985 ± 0.43737094001280385
Chi2 = 17.996811244687954
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      17.9968
NDf                       =            7
Edm                       =  5.75478e-08
NCalls                    =          113
const                     =       44.719   +/-   6.62893     
mu                        =      1.67146   +/-   0.455909    
sigma                     =      4.10821   +/-   0.437371     	 (limited)


-1.2897429538155833
0
-2.1981200265130023
60
-6.974478377646792
72
-12.390186883411818
73
-67.30271334541757
74
-77.38062014776582
75
-29.52301754240967
76
-11.4043879478527
77
-2.8889402120839622
79
-3.2680064771844717
83
-8.89958032336009
91
-7.715582162739926
98
-1.1892462703574989
102
-5.657565313271927
103
-1.8649536280536496
104
-9.61953897650405
106
-11.

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [ ]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [ ]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[-4.41423285e+01 -5.73789566e+00 -2.87768181e+01 -2.00162202e+01
 -1.59262745e+01 -1.06256629e+01 -1.29214959e+01 -5.04533473e+00
 -1.06310211e+01 -9.34107928e+00  4.67606787e+00 -8.30515499e+00
 -1.81302994e+00 -5.65716464e+00  9.38880189e+00  1.10980898e+01
  1.06233672e+01  7.86925652e+00  9.90917728e+00  6.56927354e+00
  1.14469560e+01  1.00151414e+01  1.47002647e+01  1.07249366e+01
  9.61074345e+00  7.55607158e+00  6.36327800e+00  1.12821961e+01
  1.26622123e+01  1.01835500e+01  1.52776568e+01  1.24598338e+01
  1.22077495e+01  1.23936544e+01  1.32106742e+01  1.33331940e+01
  1.49426058e+01  1.45410202e+01  1.68220772e+01  1.35939218e+01
  1.58247778e+01  1.48971425e+01  9.87582007e+00  8.09156417e+00
  1.31735715e+01  1.06220024e+01  1.19327608e+01  1.40704095e+01
  1.20910589e+01  1.24783240e+01  1.47318766e+01  1.09101256e+01
  1.34138095e+01  1.23421423e+01  1.41474610e+01  1.34315940e+01
  1.38481834e+01  1.08725076e+01  1.17486053e+01  1.22667500e+01
 -8.34154206e-01  1.89503

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
